In [2]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: d:\Pytnon_scripts\start_vector
✅ Sys path: ['d:\\Pytnon_scripts\\start_vector', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']...


In [3]:
from src_oop.core.database import Database
from src_oop.core.my_gspread import GoogleTabs
from src_oop.storage.google_sheets.google_sheets import delivery_calculation_china, annual_procurement_plan
import pandas as pd
from datetime import datetime, timedelta
import gspread

python-dotenv could not parse statement starting at line 13
python-dotenv could not parse statement starting at line 13


In [ ]:
# Создаем соединение с гугл-таблицей из таблицы из которой забираем данные
google_table_from = delivery_calculation_china.get("title")
table_sheet_from = delivery_calculation_china.get("white_orders_sheet")
google_connect_from = GoogleTabs(table_title=google_table_from, sheet_title=table_sheet_from)

# Считываем данные из таблицы из которой забираем данные
table_sheet_from_data = google_connect_from.sheet_title.get_all_values()
#  Создаем датафрейм
df = pd.DataFrame(table_sheet_from_data[4:], # Данные начиная с 5-й строки
                   columns=table_sheet_from_data[3]) # Названия колонок с 4-й строки

# Выбираем нужные колонки
# choosen_columns = ["wild", "Статус", "Кол-во к заказу"]
cancel_statuses = ["отмена", "в планах", "прибыло"]
# Фильтрация
df_short = df.loc[~df['Статус'].isin(cancel_statuses)]
df_short["updatet_at"] = (datetime.now()).strftime("%Y-%m-%d %H:%M:%S")

✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы белые ТЕСТ


ValueError: 160 columns passed, passed data had 162 columns

In [8]:
# Определяем таблицу и лист для вставки данных
google_table_to = annual_procurement_plan.get("title")
table_sheet_to = annual_procurement_plan.get("orders_sheet")


# Создаем соединение с гугл-таблицей из таблицы из которой забираем данные
try:
    # Создаем соединение с гугл-таблицей
    google_connect_to = GoogleTabs(table_title=google_table_to, sheet_title=table_sheet_to)
    # Вставляем данные в гугл-таблицу
    google_connect_to._update_df_in_google(df_short, google_connect_to.sheet_title)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Не найдена таблица {google_table_to}")
except gspread.exceptions.WorksheetNotFound as e:
    print(f"Не найден лист {table_sheet_to} в таблице {google_table_to}")
except StopIteration:
    print(f"Не найден лист {table_sheet_to} в таблице {google_table_to}")
except RuntimeError as e:
    print(f"Ошибка подключения: {e}")

✅ Успешное подключение к Годовой план закупа 2026 -> БД_ЗАКАЗЫ
Данные успешно перезаписаны на лист.


In [ ]:
engine = Database.get_engine()

query = """ 
    WITH exp AS (
    SELECT
        end_date,
        -- Individual expense columns
        SUM(CASE WHEN type='Расходы офис' THEN value ELSE 0 END) AS "Расходы офис",
        SUM(CASE WHEN type='Услуги подбора персонала' THEN value ELSE 0 END) AS "Услуги подбора персонала",
        SUM(CASE WHEN type='Арендные платежи (офис)' THEN value ELSE 0 END) AS "Арендные платежи (офис)",
        SUM(CASE WHEN type='Услуги связи (интернет, телефон)' THEN value ELSE 0 END) AS "Услуги связи (интернет, телефон)",
        SUM(CASE WHEN type='Стоянка' THEN value ELSE 0 END) AS "Стоянка",
        SUM(CASE WHEN type='Эксплуатация здания' THEN value ELSE 0 END) AS "Эксплуатация здания",
        SUM(CASE WHEN type='Коммунальные платежи' THEN value ELSE 0 END) AS "Коммунальные платежи",
        SUM(CASE WHEN type='Расчету с поставщиком материалы' THEN value ELSE 0 END) AS "Расчету с поставщиком материалы",
        SUM(CASE WHEN type='Карго' THEN value ELSE 0 END) AS "Карго",
        SUM(CASE WHEN type='Консультационные услуги по оформлении сделки' THEN value ELSE 0 END) AS "Консультационные услуги по оформлении сделки",
        SUM(CASE WHEN type='Расходы склад' THEN value ELSE 0 END) AS "Расходы склад",
        SUM(CASE WHEN type='Транспортные расходы, гсм' THEN value ELSE 0 END) AS "Транспортные расходы, ГСМ",
        SUM(CASE WHEN type='Парковка' THEN value ELSE 0 END) AS "Парковка",
        SUM(CASE WHEN type='Обслуживание а/м' THEN value ELSE 0 END) AS "Обслуживание а/м",
        SUM(CASE WHEN type='Арендные платежи (склад)' THEN value ELSE 0 END) AS "Арендные платежи (склад)",
        SUM(CASE WHEN type='Страховка' THEN value ELSE 0 END) AS "Страховка",
        SUM(CASE WHEN type='ПО, сервисы, обслуживание ПО' THEN value ELSE 0 END) AS "ПО, сервисы, обслуживание ПО",
        SUM(CASE WHEN type='Реклама/продвижение на площадках МП' THEN value ELSE 0 END) AS "Реклама/продвижение на площадках МП",
        SUM(CASE WHEN type='Вб смена номера, перенос карточек' THEN value ELSE 0 END) AS "Вб смена номера, перенос карточек",
        SUM(CASE WHEN type='Услуги банка' THEN value ELSE 0 END) AS "Услуги банка",
        SUM(CASE WHEN type='Налог: НДФЛ' THEN value ELSE 0 END) AS "Налог: НДФЛ",
        SUM(CASE WHEN type='Налог: УСН' THEN value ELSE 0 END) AS "Налог: УСН",
        SUM(CASE WHEN type='Налог: СВ' THEN value ELSE 0 END) AS "Налог: СВ",
        SUM(CASE WHEN type='Налог: прочее' THEN value ELSE 0 END) AS "Налог: прочее",
        SUM(CASE WHEN type='Налог:НДС' THEN value ELSE 0 END) AS "Налог: НДС",
        SUM(CASE WHEN type='Штрафы, взыскания' THEN value ELSE 0 END) AS "Штрафы, взыскания",
        SUM(CASE WHEN type='Проценты кредит ВБ' THEN value ELSE 0 END) AS "Проценты кредит ВБ",
        SUM(CASE WHEN type='Проценты СИМПЛФИНАНС ООО МКК' THEN value ELSE 0 END) AS "Проценты СИМПЛФИНАНС",
        SUM(CASE WHEN type='Займ ВБ Проценты' THEN value ELSE 0 END) AS "Займ ВБ: проценты",
        SUM(CASE WHEN type='Кредит сберпроценты' THEN value ELSE 0 END) AS "Кредит Сбер: проценты",
        SUM(CASE WHEN type='Рови факторинг: проценты' THEN value ELSE 0 END) AS "Рови факторинг: проценты",
        SUM(CASE WHEN type='Рови факторинг:пени' THEN value ELSE 0 END) AS "Рови факторинг: пени",
        SUM(CASE WHEN type='Лизинг, покупка оборудования, помещений' THEN value ELSE 0 END) AS "Лизинг, покупка оборудования, помещений",
        SUM(CASE WHEN type='Заработная плата' THEN value ELSE 0 END) AS "Заработная плата",
        -- Sum of all above as 'Расходы компании'
        SUM(
            CASE WHEN type IN (
                'Расходы офис', 'Услуги подбора персонала', 'Арендные платежи (офис)',
                'Услуги связи (интернет, телефон)', 'Стоянка', 'Эксплуатация здания',
                'Коммунальные платежи', 'Расчету с поставщиком материалы', 'Карго',
                'Консультационные услуги по оформлении сделки', 'Расходы склад',
                'Транспортные расходы, гсм', 'Парковка', 'Обслуживание а/м',
                'Арендные платежи (склад)', 'Страховка', 'ПО, сервисы, обслуживание ПО',
                'Реклама/продвижение на площадках МП', 'Вб смена номера, перенос карточек',
                'Услуги банка', 'Налог: НДФЛ', 'Налог: УСН', 'Налог: СВ', 'Налог: прочее',
                'Налог:НДС', 'Штрафы, взыскания', 'Проценты кредит ВБ', 'Проценты СИМПЛФИНАНС ООО МКК',
                'Займ ВБ Проценты', 'Кредит сберпроценты', 'Рови факторинг: проценты',
                'Рови факторинг:пени', 'Лизинг, покупка оборудования, помещений', 'Заработная плата'
            ) THEN value ELSE 0 END
        ) AS "Расходы компании"
    FROM expenses
    GROUP BY end_date
),
wfrm_agg AS (
    SELECT
        date_to,
        SUM("Выручка") AS "Выручка",
        SUM("Удержания") AS "Удержания",
        SUM("Логистика") AS "Логистика",
        SUM("Перечисления по кредиту") AS "Перечисления по кредиту",
        SUM("ВП после ВБ") AS "ВП после ВБ"
    FROM weekly_fin_reports_mv
    GROUP BY date_to
)
SELECT
    wfrm.*,
    w."Удержания" / NULLIF(w."Выручка", 0) AS "Удержания_pct",
    w."Логистика" / NULLIF(w."Выручка", 0) AS "Логистика_pct",
    w."Перечисления по кредиту" / NULLIF(w."Выручка", 0) AS "Перечисления по кредиту_pct",
    (w."ВП после ВБ" - e."Расходы компании") AS "Чистая прибыль",
    (w."ВП после ВБ" - e."Расходы компании")
        / NULLIF(w."Выручка", 0) AS "Чистая прибыль_pct",
    e.*
FROM weekly_fin_reports_mv wfrm
LEFT JOIN wfrm_agg w
    ON wfrm.date_to = w.date_to
LEFT JOIN exp e
    ON wfrm.date_to = e.end_date
ORDER BY wfrm.date_to DESC;
    """

In [5]:
df = pd.read_sql(query, engine)

In [7]:
df.columns

Index(['date_to', 'Комиссия ВБ', 'Комиссия ВБ pct', 'К перечислению',
       'Логистика', 'Итого к оплате', 'Выручка', 'Розничная цена со скидкой',
       'Штрафы', 'Хранение', 'Удержания', 'Платная приемка',
       'Перечисления по кредиту', 'К клиенту при отмене',
       'От клиента при отмене', 'От клиента при возврате',
       'К клиенту при продаже', 'Закупочная стоимость продаж',
       'Закупочная стоимость возвратов', 'Закупочная стоимость',
       'Наша доля до вычета себестоимости', 'ВП после ВБ', 'ВП после ВБ pct',
       'Удержания_pct', 'Логистика_pct', 'Перечисления по кредиту_pct',
       'Чистая прибыль', 'Чистая прибыль_pct', 'end_date', 'Расходы офис',
       'Услуги подбора персонала', 'Арендные платежи (офис)',
       'Услуги связи (интернет, телефон)', 'Стоянка', 'Эксплуатация здания',
       'Коммунальные платежи', 'Расчету с поставщиком материалы', 'Карго',
       'Консультационные услуги по оформл', 'Расходы склад',
       'Транспортные расходы, ГСМ', 'Парковка'